# Linear Regression: Fitting, Reading, and Optimizing a Linear Model

Lecture 3 derived the least-squares estimator on the board. This notebook runs
the same derivation on data, then pushes on it: what a coefficient means when
other columns are present, what residuals say when the model is wrong, and how
to fit the model when the closed form is too expensive to compute.

| section | the question it answers | data |
|---|---|---|
| 1. One line by hand | What does least squares compute, and how do we check it? | `Advertising` |
| 2. Three channels at once, and collinearity | What does a coefficient mean when the columns are correlated? | `Advertising` |
| 3. A time series | What do residuals tell us when the model is missing something? | NYT Covid counts |
| 4. Gradient descent | How do we fit when solving the normal equations is too costly? | `Advertising`, simulated |
| 5. How accurately to optimize | Does running the optimizer longer buy a better model? | simulated |

Two conventions run through the notebook. The quality of a fit is measured by
its **loss**, the average squared residual

$$L(\beta)=\frac1n\sum_{i=1}^n\big(y_i-x_i^\top\beta\big)^2,$$

which is the objective least squares minimizes. And every fit is checked two ways:
a formula against a library call, or a computation against a geometric
identity. A number you can reproduce two ways is a number you can trust.

The data sets are `Advertising` from *An Introduction to Statistical Learning*
(James, Witten, Hastie, Tibshirani, and Taylor, 2023), the 200 markets from the
slides, and the daily United States Covid-19 counts compiled by *The New York
Times*. Both files sit beside this notebook; the first cell fetches them when
the notebook is opened in Colab.

In [ ]:
# Colab opens this notebook on its own, so fetch the data files that sit
# beside it in the course repository. Running locally, this does nothing.
from pathlib import Path as _Path
import urllib.request as _req
_RAW = 'https://raw.githubusercontent.com/ZhuoranYang/sds265-fall26/main/demos/03-linear-regression/'
for _name in ['Advertising.csv', 'nyt-us-covid.csv']:
    if not _Path(_name).exists():
        _req.urlretrieve(_RAW + _name, _name)

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The course palette, so the notebook figures match the slides.
BLUE, CLAY, SAGE, MUTED = '#00356B', '#7A3E47', '#5B7864', '#66727E'
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})
rng = np.random.default_rng(265)


def fit_ls(X, y):
    "Least squares by the normal equations: solve X'X beta = X'y."
    return np.linalg.solve(X.T @ X, X.T @ y)


def loss(X, y, beta):
    "The least-squares loss: the average squared residual."
    return np.mean((y - X @ beta) ** 2)

---

## 1. One line by hand

### The data

Each row of `Advertising` is one market. The three budgets are in thousands of
dollars and `sales` is in thousands of units, so a slope of $0.05$ reads as
"fifty units per thousand dollars".

In [ ]:
ad = pd.read_csv('Advertising.csv', index_col=0)
print(ad.shape)
ad.head()

In [ ]:
ad.describe().round(1)

The spreads differ by a lot: `TV` runs from $0$ to about $300$ while `radio`
stops near $50$. That looks harmless now. It is the reason gradient descent
zigzags in Section 4.

### The formula from the board

For one feature the least-squares line is

$$\widehat{\beta}_1=\frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sum_i (x_i-\bar x)^2}
=\frac{S_{xy}}{S_{xx}},\qquad \widehat{\beta}_0=\bar y-\widehat{\beta}_1\bar x .$$

**Reading the code.** `x` and `y` are the `TV` and `sales` columns as NumPy
arrays. The two sums are computed exactly as written, and the printout matches
the worked example on the slides: $\bar x=147.04$, $\bar y=14.02$,
$S_{xy}=69{,}727.6$, $S_{xx}=1{,}466{,}819$.

In [ ]:
x = ad['TV'].to_numpy()
y = ad['sales'].to_numpy()

xbar, ybar = x.mean(), y.mean()
S_xy = ((x - xbar) * (y - ybar)).sum()
S_xx = ((x - xbar) ** 2).sum()

beta1 = S_xy / S_xx
beta0 = ybar - beta1 * xbar
print(f'xbar = {xbar:.2f}   ybar = {ybar:.2f}')
print(f'S_xy = {S_xy:,.1f}   S_xx = {S_xx:,.0f}')
print(f'beta0 = {beta0:.3f}   beta1 = {beta1:.4f}')

### The same answer through the normal equations

The matrix form of the same problem stacks a column of ones next to `TV`, so
that $X\beta=\beta_0+\beta_1x$ for every row at once, and solves
$X^\top X\beta=X^\top y$. That is what `fit_ls` does, and it is the route that
scales to many features.

**Reading the code.** `np.c_` glues columns side by side. The first assertion
says the two routes agree to floating-point precision. The second is the
geometric fact from the lecture: at the minimizer the residual
$r=y-X\widehat{\beta}$ is **orthogonal to every column of $X$**, so
$X^\top r=0$. Both are the kind of check to run whenever you fit anything by
hand.

In [ ]:
X = np.c_[np.ones_like(x), x]
beta = fit_ls(X, y)
assert np.allclose(beta, [beta0, beta1])

resid = y - X @ beta
print('X.T @ resid =', X.T @ resid)
assert np.allclose(X.T @ resid, 0, atol=1e-8)

print(f'loss = {loss(X, y, beta):.2f}')

The loss is about $10.5$, in squared thousands of units. Here is the fit, with
every residual drawn as a vertical segment. The line is the one that makes the
average of the squared segment lengths smallest, and that average is the loss.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
grid = np.linspace(x.min(), x.max(), 2)
for xi, yi, fi in zip(x, y, X @ beta):
    ax.plot([xi, xi], [fi, yi], color=MUTED, lw=.5, zorder=1)
ax.scatter(x, y, s=12, color=CLAY, zorder=3)
ax.plot(grid, beta[0] + beta[1] * grid, color=BLUE, lw=2, zorder=4)
ax.set(xlabel='TV budget (thousands of dollars)', ylabel='sales (thousands of units)',
       title=f'sales = {beta[0]:.3f} + {beta[1]:.4f} TV')
plt.show()

**Reading the slope.** Each extra $\$1{,}000$ of TV budget goes with
$0.0475\times1000\approx48$ more units sold. Turned around, selling $1{,}000$
more units needs about $1000/0.0475\approx\$21{,}000$ of extra TV budget.

---

## 2. Three channels at once, and collinearity

Fitting each channel on its own gives three slopes. Fitting all three together
gives three different slopes, and one of them changes sign. Understanding why
is the point of this section.

**Reading the code.** The loop fits `sales` on one budget at a time. The last
block builds the full design matrix, four columns including the intercept, and
fits once. `loss` measures each fit the same way.

In [ ]:
channels = ['TV', 'radio', 'newspaper']
alone = {}
for name in channels:
    Xi = np.c_[np.ones(len(ad)), ad[name]]
    b = fit_ls(Xi, y)
    alone[name] = b[1]
    print(f'{name:10s} alone:  slope {b[1]: .4f}   loss {loss(Xi, y, b):.2f}')

X_all = np.c_[np.ones(len(ad)), ad[channels].to_numpy()]
beta_all = fit_ls(X_all, y)
print(f'\nall three together:  loss {loss(X_all, y, beta_all):.2f}')
pd.DataFrame({'alone': [np.nan] + [alone[c] for c in channels],
              'together': beta_all}, index=['intercept'] + channels).round(4)

Two things happened. The loss fell from about $10.5$ to about $2.8$: the three
budgets together predict sales far better than TV alone. And `newspaper` went from a clear positive slope on its own to a
coefficient indistinguishable from zero.

The correlation matrix is the first clue.

In [ ]:
ad.corr().round(3)

`radio` and `newspaper` correlate at $0.35$: markets that buy radio also buy
newspaper. So when newspaper is the only column, its slope also picks up the
sales that radio produced in those same markets. Once radio is in the model,
radio takes that credit back, and newspaper is left with nothing.

### Checking the argument

The explanation above is a claim, and we can validate it: look at the
contribution of newspaper with the contribution of radio removed. Regress
`newspaper` on `radio` and keep the residual,

$$\widetilde{x}_i = \text{newspaper}_i - (\widehat{\gamma}_0 + \widehat{\gamma}_1\,\text{radio}_i),$$

the part of each market's newspaper budget that its radio budget cannot
explain. By construction
$\widetilde{x}$ is uncorrelated with radio. Then regress sales on
$\widetilde{x}$ alone. If newspaper was only taking credit for radio, this
slope should be near zero. It is, and more than that: it equals the newspaper
coefficient in `sales ~ radio + newspaper` **exactly**. That is what the
coefficient in a multiple regression computes, and what "holding radio fixed"
means. Partial out TV as well and the coefficient from the full table comes
back.

**Reading the code.** `gamma` is the two-coefficient fit of newspaper on radio.
`news_tilde` is its residual. The two printed slopes are computed by different
routes and agree to four decimals; then the same is done with TV included.

In [ ]:
one = np.ones(len(ad))
tv, radio, news = ad['TV'].to_numpy(), ad['radio'].to_numpy(), ad['newspaper'].to_numpy()

gamma = fit_ls(np.c_[one, radio], news)             # newspaper ~ radio
news_tilde = news - (gamma[0] + gamma[1] * radio)   # what radio cannot explain
print(f'newspaper = {gamma[0]:.2f} + {gamma[1]:.4f} radio + residual')
print(f'corr(news_tilde, radio) = {np.corrcoef(news_tilde, radio)[0, 1]:.1e}')

slope_tilde = fit_ls(np.c_[one, news_tilde], y)[1]
slope_two = fit_ls(np.c_[one, radio, news], y)[2]
print(f'\nslope of sales on news_tilde            : {slope_tilde:.4f}')
print(f'newspaper coef in sales ~ radio + news  : {slope_two:.4f}')

gamma2 = fit_ls(np.c_[one, tv, radio], news)        # partial out TV as well
news_tilde2 = news - np.c_[one, tv, radio] @ gamma2
print(f'\nslope of sales on news_tilde2           : {fit_ls(np.c_[one, news_tilde2], y)[1]:.4f}')
print(f'newspaper coef in sales ~ TV+radio+news : {beta_all[3]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, xx, label, title in [
        (axes[0], news, 'newspaper budget', 'newspaper alone'),
        (axes[1], news_tilde,
         r'$\widetilde{x}$ = newspaper $-$ ($\widehat{\gamma}_0+\widehat{\gamma}_1\,$radio)',
         'radio accounted for')]:
    c = fit_ls(np.c_[one, xx], y)
    g = np.linspace(xx.min(), xx.max(), 2)
    ax.scatter(xx, y, s=10, color=BLUE, alpha=.6)
    ax.plot(g, c[0] + c[1] * g, color=CLAY, lw=2)
    ax.set(xlabel=label, title=f'{title}: slope {c[1]:.4f}')
axes[0].set_ylabel('sales')
plt.show()

This is **collinearity**: two columns correlated enough that part of what one
explains, the other explains too. Least squares can credit the shared part to
only one of them, so the coefficient on newspaper answers "what does newspaper
add beyond radio", and here the answer is nothing.

With a correlation of $0.35$ the answer is at least stable. When two columns are
nearly identical, $\widetilde{x}$ has almost no spread left and the slope on it
is decided by noise: the coefficients swing wildly from one sample to the next
while the fitted values barely move. Lecture 4 shows how to detect this through
the condition number and how ridge regression tames it.

---

## 3. A time series: Covid-19 case counts

The *New York Times* compiled daily Covid-19 counts for the United States from
January 2020 until March 2023, when it stopped. The file beside this notebook
has the national totals: one row per day with the **cumulative** number of
cases and deaths.

Linear regression is often the first thing people fit to a series like this,
usually to answer "which way is it going?". The exercise here is to fit that
line honestly and then read the residuals, because they say exactly what the
model is missing.

**Reading the code.** `parse_dates` makes the index a date, so pandas can do
calendar arithmetic. New cases per day are the day-to-day differences of the
cumulative count, `diff()`. The seven-day rolling mean is the smooth curve the
newspapers plot.

In [ ]:
covid = pd.read_csv('nyt-us-covid.csv', parse_dates=['date'], index_col='date')
covid['new_cases'] = covid['cases'].diff()
covid['avg7'] = covid['new_cases'].rolling(7).mean()
covid = covid.dropna()
print(covid.index.min().date(), 'to', covid.index.max().date(), '-', len(covid), 'days')

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(covid.index, covid['new_cases'] / 1e3, width=1, color=CLAY, alpha=.3)
ax.plot(covid.index, covid['avg7'] / 1e3, color=CLAY, lw=1.8)
ax.set(ylabel='new cases per day (thousands)', title='United States, reported cases')
plt.show()

### A line through one summer

Zoom in on the two months after the summer 2022 wave peaked, when cases were
falling. Use the day number $t=0,1,2,\dots$ as the feature and new cases as
the response, exactly as in Section 1 with one column.

In [ ]:
window = covid.loc['2022-07-20':'2022-09-20']
t = np.arange(len(window), dtype=float)
yc = window['new_cases'].to_numpy()

X_line = np.c_[np.ones_like(t), t]
b_line = fit_ls(X_line, yc)
print(f'slope: {b_line[1]:,.0f} cases per day per day')
print(f'loss: {loss(X_line, yc, b_line):.3g}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.scatter(window.index, yc / 1e3, s=14, color=BLUE)
ax.plot(window.index, (X_line @ b_line) / 1e3, color=CLAY, lw=2)
ax.set(ylabel='new cases (thousands)', title='a line through 63 days')
fig.autofmt_xdate()
plt.show()

The slope is negative, as expected. But the fit is poor in a very particular
way: the points sit in two bands, one well above the line and one well below,
and the low band recurs every few days. The residuals show it. Group them by
the day of the week.

In [ ]:
resid_line = yc - X_line @ b_line
day_of_week = window.index.dayofweek      # Monday = 0, ..., Sunday = 6
names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
by_day = pd.Series(resid_line).groupby(day_of_week).mean()

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.bar(names, by_day / 1e3, color=[BLUE if v > 0 else CLAY for v in by_day])
ax.axhline(0, color=MUTED, lw=.8)
ax.set(ylabel='mean residual (thousands)', title='the line misses by day of the week')
plt.show()

Reported cases are not infections. Test sites and health departments report
less on weekends and catch up midweek, so Saturday and Sunday land far below
the line and Wednesday far above it. The pattern is not noise: it is a
**feature the model does not have**.

### Giving the model the feature

Two responses are common. One smooths the data first, which is what the
seven-day average does, and then fits the line to the smooth curve. That works,
but the smoothing is a modelling decision hidden inside the preprocessing.

The other keeps the raw counts and puts the day of the week into $X$. Write
$d(t)\in\{1,\dots,7\}$ for the day of the week of day $t$, Monday to Sunday.
The model is

$$y_t=\sum_{k=1}^{7}\alpha_k\,\mathbf{1}\{d(t)=k\}+\beta\,t+\varepsilon_t,$$

so the row of $X$ for day $t$ is the seven indicators followed by $t$,
$x_t=(\mathbf{1}\{d(t)=1\},\dots,\mathbf{1}\{d(t)=7\},\,t)$, and the
coefficient vector is $(\alpha_1,\dots,\alpha_7,\beta)$. Exactly one
indicator is on for each day, so the seven columns replace the intercept: each
day of the week gets its own intercept $\alpha_k$, and all seven share one
slope $\beta$. The model is still linear in the coefficients, so least squares
fits it exactly as before. This is the first example in the course of
**feature engineering**: the linear model is linear in the coefficients, and the
columns of $X$ can be anything you can compute from the data.

**Reading the code.** `np.eye(7)[day_of_week]` picks row `day_of_week[i]` of the
identity matrix for each day, which is its indicator vector. `np.c_` appends the
time column.

In [ ]:
D = np.eye(7)[day_of_week]                # one indicator column per day of the week
X_week = np.c_[D, t]
b_week = fit_ls(X_week, yc)

print(f'shared slope: {b_week[7]:,.0f} cases per day per day')
print(f'loss: {loss(X_week, yc, b_week):.3g}  (was {loss(X_line, yc, b_line):.3g})')
print('\nintercepts by day of the week (thousands):')
print(pd.Series(b_week[:7] / 1e3, index=names).round(1).to_string())

resid_week = yc - X_week @ b_week
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].scatter(window.index, yc / 1e3, s=14, color=BLUE)
axes[0].plot(window.index, (X_week @ b_week) / 1e3, color=CLAY, lw=1.4)
axes[0].set(ylabel='new cases (thousands)', title='one slope, seven intercepts')
axes[1].plot(window.index, resid_line / 1e3, color=MUTED, lw=1, label='line only')
axes[1].plot(window.index, resid_week / 1e3, color=CLAY, lw=1.6, label='with day-of-week indicators')
axes[1].axhline(0, color=MUTED, lw=.8)
axes[1].set(ylabel='residual (thousands)', title='residuals over time')
axes[1].legend(frameon=False)
fig.autofmt_xdate()
plt.show()

The slope hardly moved: the trend was there all along. What changed is how well
the model accounts for the data. The loss fell by a factor of about ten, and the
weekly sawtooth in the residuals is gone. Each intercept $\alpha_k$ is where
that day's line sits at $t=0$; the Saturday and Sunday intercepts sit about
$100{,}000$ below Monday's, which is the reporting gap made explicit.

One more check connects back to Section 1. With an indicator column for each
day of the week, the orthogonality condition $X^\top r=0$ says the residuals of
every Monday sum to exactly zero, and likewise for each other day, so the bar
chart of mean residual by day of the week is flat by construction and no longer
tells us anything. That is why the panel above shows residuals over time
instead.

### What the line says about the future

The slope answers "how fast were cases falling *in this window*". It is
tempting to run the line forward. Do it for the next sixty days and compare with
what was reported.

**Reading the code.** The future design matrix uses the same day-of-week indicators
and continues the day count past the window. `covid.loc` pulls what actually
happened.

In [ ]:
future = covid.loc['2022-09-21':'2022-11-19']
t_future = np.arange(len(window), len(window) + len(future), dtype=float)
X_future = np.c_[np.eye(7)[future.index.dayofweek], t_future]
pred = X_future @ b_week

fig, ax = plt.subplots(figsize=(8, 3))
ax.scatter(window.index, yc / 1e3, s=10, color=BLUE, label='fitted window')
ax.scatter(future.index, future['new_cases'] / 1e3, s=10, color=SAGE, label='what happened')
ax.plot(future.index, pred / 1e3, color=CLAY, lw=2, label='the line, continued')
ax.axhline(0, color=MUTED, lw=.8)
ax.set(ylabel='new cases (thousands)')
ax.legend(frameon=False)
fig.autofmt_xdate()
plt.show()

print(f"final week of the forecast: the line predicts {pred[-7:].mean():,.0f} cases per day; "
      f"reported average {future['new_cases'].iloc[-7:].mean():,.0f}")

By mid-November the line predicts a negative number of cases. Reported cases
levelled off near forty thousand a day instead, weekends included. Nothing in
the fit was wrong:
the model class is straight lines, a straight line with negative slope crosses
zero, and the data inside the window gave no reason to expect a floor.

Three things to carry forward from this section.

- **Structured residuals mean a missing feature.** Plotting residuals against
  anything you know about the rows (here the day of the week) is the quickest
  diagnostic there is.
- **Linear in $\beta$ is not linear in the data.** Indicators, powers, and
  products of the raw columns are all fair game, and Lecture 6 takes this much
  further.
- **A fit describes the window it was fitted on.** Extrapolation is a modelling
  assumption the data cannot check.

---

## 4. Gradient descent

Every fit so far solved the normal equations, which costs about $np^2$
operations to form $X^\top X$ and $p^3$ to solve. With $p$ in the thousands, as
it will be for neural networks, that is too much. The alternative is to walk
downhill on the loss

$$L(\beta)=\frac1n\|X\beta-y\|_2^2,\qquad \nabla L(\beta)=\frac2n X^\top(X\beta-y),$$

taking a step of size $\eta$ against the gradient each time. This $L$ is the
same loss as before, written with a norm. The whole
algorithm is the function below. `path` keeps every iterate so we can plot the
route afterwards.

In [ ]:
def gradient_descent(X, y, lr, steps, beta0=None):
    beta = np.zeros(X.shape[1]) if beta0 is None else np.array(beta0, float)
    path = [beta.copy()]
    n = len(y)
    for _ in range(steps):
        grad = 2.0 / n * X.T @ (X @ beta - y)
        beta = beta - lr * grad
        path.append(beta.copy())
    return np.array(path)

### The learning rate

Lecture 5 will prove that on a quadratic objective gradient descent converges
exactly when $0<\eta<2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest
eigenvalue of the Hessian $\tfrac2nX^\top X$. The boundary is real, and the
next two cells show it.

**Reading the code.** `Z` holds `TV` and `radio` standardized to mean zero and
standard deviation one, and `yc` is sales with its mean removed, so no
intercept column is needed. `np.linalg.eigvalsh` returns the eigenvalues of a
symmetric matrix.

In [ ]:
Z = np.c_[(tv - tv.mean()) / tv.std(), (radio - radio.mean()) / radio.std()]
ys = y - y.mean()

H = 2.0 / len(ys) * Z.T @ Z
lam = np.linalg.eigvalsh(H)
print('Hessian eigenvalues:', lam.round(3))
print(f'largest stable learning rate  2 / lambda_max = {2 / lam.max():.3f}')

In [ ]:
best = fit_ls(Z, ys)
floor = loss(Z, ys, best)

fig, ax = plt.subplots(figsize=(7, 3.2))
for lr, color, label in [(0.02, SAGE, 'eta = 0.02, too small'),
                         (0.40, BLUE, 'eta = 0.40, good'),
                         (1.02 * 2 / lam.max(), CLAY, 'eta just above 2 / lambda_max')]:
    path = gradient_descent(Z, ys, lr, 60)
    excess = np.array([loss(Z, ys, b) for b in path]) - floor
    ax.semilogy(np.maximum(excess, 1e-15), color=color, lw=2, label=label)
ax.set(xlabel='iteration', ylabel='loss above the minimum')
ax.legend(frameon=False)
plt.show()

Three regimes. Too small a step converges, slowly. A good step reaches
floating-point precision in a few dozen iterations. A step just past the bound
grows without limit: each iteration overshoots the minimum by more than the
last, and the iterates leave.

### Why the features were standardized

Run the same fit on the raw scale, where TV varies over hundreds and radio over
tens, and compare the eigenvalues.

In [ ]:
Z_raw = np.c_[tv - tv.mean(), radio - radio.mean()]

for name, M in [('standardized', Z), ('raw scale', Z_raw)]:
    lam_m = np.linalg.eigvalsh(2 / len(ys) * M.T @ M)
    print(f'{name:14s} lambda_min = {lam_m.min():9.4f}   lambda_max = {lam_m.max():10.3f}'
          f'   ratio = {lam_m.max() / lam_m.min():8.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (name, M) in zip(axes, [('raw scale', Z_raw), ('standardized', Z)]):
    bstar = fit_ls(M, ys)
    lam_m = np.linalg.eigvalsh(2 / len(ys) * M.T @ M)
    span = 2.6 * np.abs(bstar).max()
    g = [np.linspace(bstar[i] - span, bstar[i] + span, 120) for i in (0, 1)]
    G0, G1 = np.meshgrid(*g)
    Rgrid = ((ys[:, None, None] - G0[None] * M[:, 0][:, None, None]
              - G1[None] * M[:, 1][:, None, None]) ** 2).mean(axis=0)
    ax.contour(G0, G1, Rgrid, levels=16, colors='#D8E0E8', linewidths=.7)
    path = gradient_descent(M, ys, 1.8 / lam_m.max(), 60,
                            beta0=(bstar[0] - .65 * span, bstar[1] + .65 * span))
    ax.plot(path[:, 0], path[:, 1], '-o', ms=2.5, lw=1, color=CLAY)
    ax.plot(*bstar, marker='*', ms=14, color=BLUE)
    ax.set(xlabel='coefficient of TV', ylabel='coefficient of radio',
           title=f'{name}: ratio {lam_m.max() / lam_m.min():.0f}')
plt.show()

On the raw scale the loss surface is a long narrow valley. A step size safe for
the steep direction is far too small for the flat one, so the iterates bounce
across the valley instead of running along it. Standardizing makes the bowl
round and the same sixty steps land on the minimum.

The ratio $\lambda_{\max}/\lambda_{\min}$ is the **condition number**. Lecture 4
shows it also controls how unstable the coefficients are, and Lecture 5 shows
it sets the number of iterations gradient descent needs.

---

## 5. How accurately to optimize

### The cost of a step

The closed form costs about $np^2$ operations. One gradient step costs about
$np$. Measure both on random data as $p$ grows.

**Reading the code.** `perf_counter` is a wall-clock timer. Each exact solve is
repeated a few times and averaged; the gradient step is averaged over fifty
iterations. The step size is tiny on purpose, so the timing loop cannot
diverge.

In [ ]:
n_big = 10_000
rows = []
for p_big in [50, 100, 200, 400, 800, 1600]:
    Xb = rng.normal(size=(n_big, p_big))
    yb = rng.normal(size=n_big)

    t0 = time.perf_counter()
    for _ in range(3):
        np.linalg.solve(Xb.T @ Xb, Xb.T @ yb)
    exact = (time.perf_counter() - t0) / 3

    theta = np.zeros(p_big)
    t0 = time.perf_counter()
    for _ in range(50):
        theta -= 1e-5 * (2 / n_big * Xb.T @ (Xb @ theta - yb))
    step = (time.perf_counter() - t0) / 50

    rows.append({'p': p_big, 'exact solve (ms)': exact * 1e3,
                 'one GD step (ms)': step * 1e3, 'GD steps per solve': exact / step})
pd.DataFrame(rows).round(2)

Read the last column. At $p=1600$ one exact solve buys dozens of gradient steps,
and the gap widens with $p$ because one cost grows like $p^2$ and the other like
$p$. Gradient descent is worth it when a modest number of steps gets close
enough. The next cell asks what "close enough" means.

### Optimization error and statistical error

The iterate after $k$ steps differs from the truth $\beta^\star$ for two
reasons,

$$\beta^{(k)}-\beta^\star
=\underbrace{(\beta^{(k)}-\widehat{\beta})}_{\text{optimization}}
+\underbrace{(\widehat{\beta}-\beta^\star)}_{\text{statistical}},$$

and only the first term shrinks when the optimizer runs longer. On real data
$\beta^\star$ is never seen, so we simulate a world where it is.

**Reading the code.** The features are correlated on purpose, through a
Cholesky factor, so the problem is not trivially easy. `beta_star` is the true
coefficient vector, `ys_sim` the noisy response, and `beta_hat` the exact
least-squares fit.

In [ ]:
n_sim, p_sim, sigma = 400, 20, 1.0
idx = np.arange(p_sim)
corr = 0.9 ** np.abs(idx[:, None] - idx[None, :])
X_sim = rng.normal(size=(n_sim, p_sim)) @ np.linalg.cholesky(corr).T
beta_star = rng.normal(size=p_sim)
y_sim = X_sim @ beta_star + sigma * rng.normal(size=n_sim)

beta_hat = fit_ls(X_sim, y_sim)
stat_err = np.linalg.norm(beta_hat - beta_star)
print(f'statistical error  ||beta_hat - beta_star|| = {stat_err:.3f}')

lr = 0.5 / np.linalg.eigvalsh(2 / n_sim * X_sim.T @ X_sim).max()
path = gradient_descent(X_sim, y_sim, lr, 900)
opt_err = np.linalg.norm(path - beta_hat, axis=1)
tot_err = np.linalg.norm(path - beta_star, axis=1)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.semilogy(opt_err, color=CLAY, lw=2, label='optimization error')
ax.semilogy(tot_err, color=BLUE, lw=2, label='total error')
ax.axhline(stat_err, color=SAGE, lw=2, ls='--', label='statistical error')
ax.set(xlabel='gradient-descent iteration', ylabel='error')
ax.legend(frameon=False)
plt.show()

The optimization error keeps falling. The total error stops the moment it
reaches the statistical error and never improves again. Past that point,
optimizing harder buys accuracy in a quantity that does not matter, and the
statistical error is set by the data, the noise, and the model class. This is
why the optimizers later in the course stop long before convergence.

---

## Try it

Each prompt changes one assumption. Decide what should happen before running
the cell.

1. **Another channel.** In Section 1 replace `TV` by `radio`. Is the
   loss larger or smaller, and does the slope match the one-at-a-time table in
   Section 2?
2. **One bad market.** Append a row to the Section 1 data with `TV = 300` and
   `sales = 100`, refit, and watch the slope. Then fit the same data with
   `sklearn.linear_model.HuberRegressor`. Squared loss charges $r^2$, so one
   residual ten times the typical size costs a hundred times as much; the Huber
   loss charges it linearly.
3. **Another window.** Move the Covid window to the Omicron rise,
   `'2021-12-01'` to `'2022-01-20'`. Does a line make sense there? Add a column
   `t ** 2` to `X_week` and see what the fit does to the residuals and to the
   sixty-day forecast.
4. **On the boundary.** In Section 4 set the learning rate to exactly
   `2 / lam.max()`. The theory says the iteration neither converges nor
   diverges. What does the loss curve do?
5. **Fewer data.** In Section 5 set `n_sim = 40`. The statistical error grows.
   How many gradient steps are now enough?

### Where this goes next

Lecture 4 picks up the two loose ends: the condition number, which decides both
how unstable coefficients are and how many iterations gradient descent needs,
and ridge regression, which improves both by adding a penalty. Lecture 5
derives the step-size rule used in Section 4 and replaces the full gradient with
a minibatch.

**Data.** `Advertising` is from *An Introduction to Statistical Learning with
Applications in Python* (James, Witten, Hastie, Tibshirani, Taylor; Springer,
2023), <https://www.statlearning.com/>. The Covid counts are from *The New York
Times*, <https://github.com/nytimes/covid-19-data>, national totals through
23 March 2023. Section 3 grows out of the `covid-trends` demo in the Spring 2026
offering of this course.